# Gallstone Data Processing in Jupyter Notebook
This notebook processes the datasets, and splitting subjects into groups for further analysis.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [5]:
class GallstoneDataProcessor:
    def __init__(self, data_path):
        """Initialize the processor with the dataset path."""
        self.data_path = data_path
        self.df = pd.read_csv(data_path, low_memory=False)
        self.filter_log = []

    def preprocess_data(self, output_path):
        """ Encode categorical variables and scale numerical features."""
        df_encoded = self.df.copy()

        # Identify categorical columns
        categorical_cols = df_encoded.select_dtypes(include=['object']).columns
        print(f"Categorical columns identified for encoding: {categorical_cols.tolist()}")

        # Create label encoders for each categorical column
        label_encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
            label_encoders[col] = le
            print(f"Encoded column '{col}' with labels: {le.classes_}")
        
        # Normalize/scalar numeric features
        # First column is the target
        feature_cols = df_encoded.columns[1:]
        target_col = df_encoded.columns[0]

        print(f"Feature columns: {list(feature_cols)}")
        print(f"Target column: {target_col}")

        # Separate features and target
        X = df_encoded[feature_cols]
        y = df_encoded[target_col]

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Convert back to DataFrame for better readability
        df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

        # Combine scaled features with target
        df_final = pd.concat([y.reset_index(drop=True), df_scaled], axis=1)
        self.df = df_final
        print("Data preprocessing completed: categorical encoding and feature scaling applied.")

        # Save the processed data
        self.df.to_csv(output_path, index=False)
        print(f"Preprocessed data saved to {output_path}.")
        

    def stratified_train_test_split(self, output_path):
        """Perform a stratified train-test split based on the 'Gallstone Status' column."""
        
        # Perform stratified split
        train_df, test_df = train_test_split(
            self.df,
            test_size=0.2,
            stratify=self.df['Gallstone Status'],
            random_state=42
        )

        # Store the resulting dataframes 

        # Assign train/test labels to df
        self.df['Set'] = np.where(self.df.index.isin(train_df.index), 'train', 'test')

        # Save to CSV
        self.df.to_csv(output_path, index=False)
        print(f"Stratified train-test split completed. Data saved to {output_path}.")
    

In [6]:
# Initiate and execute
processor = GallstoneDataProcessor('data/dataset-uci.csv')
processor.preprocess_data('data/gallstone_data_preprocessed.csv')
processor.stratified_train_test_split('data/gallstone_data_train_test_split.csv')

Categorical columns identified for encoding: []
Feature columns: ['Age', 'Gender', 'Comorbidity', 'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia', 'Diabetes Mellitus (DM)', 'Height', 'Weight', 'Body Mass Index (BMI)', 'Total Body Water (TBW)', 'Extracellular Water (ECW)', 'Intracellular Water (ICW)', 'Extracellular Fluid/Total Body Water (ECF/TBW)', 'Total Body Fat Ratio (TBFR) (%)', 'Lean Mass (LM) (%)', 'Body Protein Content (Protein) (%)', 'Visceral Fat Rating (VFR)', 'Bone Mass (BM)', 'Muscle Mass (MM)', 'Obesity (%)', 'Total Fat Content (TFC)', 'Visceral Fat Area (VFA)', 'Visceral Muscle Area (VMA) (Kg)', 'Hepatic Fat Accumulation (HFA)', 'Glucose', 'Total Cholesterol (TC)', 'Low Density Lipoprotein (LDL)', 'High Density Lipoprotein (HDL)', 'Triglyceride', 'Aspartat Aminotransferaz (AST)', 'Alanin Aminotransferaz (ALT)', 'Alkaline Phosphatase (ALP)', 'Creatinine', 'Glomerular Filtration Rate (GFR)', 'C-Reactive Protein (CRP)', 'Hemoglobin (HGB)', 'Vitamin D']
T